SDSU SCIL Census and Fema Data AI Assistant  
By Sam Jackowski  

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import plotly.express as px

## Setup

In [ ]:
PROJECT_DIR = Path(r"C:\SDSU_SCIL_CensusData")

COUNTY_PANEL_PATH = (PROJECT_DIR / "data"/ "processed" / "acs_county_year_fema_flood_risk.parquet")

GEOJSON_PATH = (PROJECT_DIR / "data" / "geo" / "us_counties_geojson.json")

print("County panel exists:", COUNTY_PANEL_PATH.exists())
print("County panel path:", COUNTY_PANEL_PATH)

print("GeoJSON exists:", GEOJSON_PATH.exists())
print("GeoJSON path:", GEOJSON_PATH)

In [ ]:

county_wide = pd.read_parquet(COUNTY_PANEL_PATH)

with open(GEOJSON_PATH, "r", encoding="utf-8") as f:
    counties_geojson = json.load(f)

county_wide["GEOID"] = county_wide["GEOID"].astype(str).str.zfill(5)
county_wide["county_name"] = county_wide["NAME"].str.split(",").str[0].str.strip()
county_wide["state_name"] = county_wide["NAME"].str.split(",").str[-1].str.strip()

id_cols = [
    "NAME",
    "county_name",
    "state_name",
    "state",
    "county",
    "year",
    "GEOID",
]

id_cols = [c for c in id_cols if c in county_wide.columns]
value_cols = [c for c in county_wide.columns if c not in id_cols]

county_long = county_wide.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name="variable",
    value_name="value",
)

county_long["GEOID"] = county_long["GEOID"].astype(str).str.zfill(5)
county_long["value"] = pd.to_numeric(county_long["value"], errors="coerce")

print("Wide:", county_wide.shape)
print("Long:", county_long.shape)
display(county_long.head())

In [ ]:
current_recipe = {
    "intent": "make_map",
    "variable": "total_population",
    "year": int(county_long["year"].max()),
    "year_start": None,
    "year_end": None,
    "animated": False,
    "state_filter": None,
    "county_filter": None,
    "color_scale": "Plasma",
    "title": "Total Population by County"
}

current_recipe

In [ ]:
def filter_county_data(county_long, recipe):
    """
    Filter county data by variable, year, state, and county.

    Supports:
    - One state: "California"
    - Multiple states: ["California", "Oregon"]
    - Nationwide: None, "", "United States", "USA", or "US"
    - One or multiple counties
    """

    df = county_long[
        (county_long["variable"] == recipe["variable"]) &
        (county_long["year"] == recipe["year"])
    ].copy()

    # -----------------------------
    # State filter
    # -----------------------------
    state_filter = recipe.get("state_filter")

    nationwide_values = {
        "",
        "united states",
        "usa",
        "us",
        "all",
        "all states",
    }

    # Convert state_filter into a list
    if state_filter is None:
        states = []

    elif isinstance(
        state_filter,
        (list, tuple, set, np.ndarray, pd.Series)
    ):
        states = [
            str(state).strip().lower()
            for state in state_filter
            if state is not None
            and str(state).strip().lower() not in nationwide_values
        ]

    else:
        state_value = str(state_filter).strip().lower()

        if state_value in nationwide_values:
            states = []
        else:
            states = [state_value]

    # Apply the state filter only when specific states were provided
    if states:
        df = df[
            df["state_name"]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(states)
        ]

    # -----------------------------
    # County filter
    # -----------------------------
    county_filter = recipe.get("county_filter")

    if county_filter:
        if isinstance(
            county_filter,
            (list, tuple, set, np.ndarray, pd.Series)
        ):
            counties = [
                str(county).strip().lower()
                for county in county_filter
                if county is not None and str(county).strip()
            ]
        else:
            counties = [
                str(county_filter).strip().lower()
            ]

        if counties:
            county_match = pd.Series(
                False,
                index=df.index
            )

            if "county" in df.columns:
                county_match = county_match | (
                    df["county"]
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    .isin(counties)
                )

            if "county_name" in df.columns:
                county_match = county_match | (
                    df["county_name"]
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    .isin(counties)
                )

            df = df[county_match]

    # -----------------------------
    # Numeric cleanup
    # -----------------------------
    df["value"] = pd.to_numeric(
        df["value"],
        errors="coerce"
    )

    df = df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    df = df.dropna(
        subset=["value"]
    )

    return df

In [ ]:
def get_variable_label(variable):
    labels = {
        "total_population": "Total Population",
        "median_age": "Median Age",
        "employment_rate": "Employment Rate",
        "unemployment_rate": "Unemployment Rate",
        "poverty_rate": "Poverty Rate",
        "median_household_income": "Median Household Income",
        "median_home_value": "Median Home Value",
        "median_gross_rent": "Median Gross Rent",
        "less_than_high_school_pct": "Less Than High School",
        "high_school_grad_pct": "High School Graduate",
        "bachelors_or_higher_pct": "Bachelor's or Higher",
    }

    return labels.get(variable, variable.replace("_", " ").title())


def get_legend_settings(variable, transform="none"):
    percent_vars = {
        "poverty_rate",
        "unemployment_rate",
        "employment_rate",
        "less_than_high_school_pct",
        "high_school_grad_pct",
        "bachelors_or_higher_pct",
    }

    dollar_vars = {
        "median_household_income",
        "median_home_value",
        "median_gross_rent",
    }


    if variable in percent_vars:
        return {
            "title": "%",
            "tickformat": ".0f",
            "hover_format": ":.1f",
        }

    if variable in dollar_vars:
        return {
            "title": "$",
            "tickformat": "$,.0f",
            "hover_format": "$,.0f",
        }

    if variable == "total_population":
        return {
            "title": "People",
            "tickformat": ",.0f",
            "hover_format": ":,.0f",
        }

    if variable == "median_age":
        return {
            "title": "Years",
            "tickformat": ".1f",
            "hover_format": ":.1f",
        }

    return {
        "title": get_variable_label(variable),
        "tickformat": ",.2f",
        "hover_format": ":,.2f",
    }

In [ ]:
def format_value_for_hover(value, variable):
    if pd.isna(value):
        return "N/A"

    percent_vars = {
        "poverty_rate",
        "unemployment_rate",
        "employment_rate",
        "less_than_high_school_pct",
        "high_school_grad_pct",
        "bachelors_or_higher_pct",
    }

    dollar_vars = {
        "median_household_income",
        "median_home_value",
        "median_gross_rent",
    }

    if variable in percent_vars:
        return f"{value:.1f}%"

    if variable in dollar_vars:
        return f"${value:,.0f}"

    if variable == "total_population":
        return f"{value:,.0f} people"

    if variable == "median_age":
        return f"{value:.1f} years"

    return f"{value:,.2f}"

In [ ]:
import numpy as np
import pandas as pd


def nice_round(value):
    """
    Round numbers to clean map legend values.
    """
    if pd.isna(value):
        return value

    value = float(value)

    if abs(value) >= 100000:
        return round(value / 10000) * 10000
    elif abs(value) >= 10000:
        return round(value / 1000) * 1000
    elif abs(value) >= 1000:
        return round(value / 100) * 100
    elif abs(value) >= 100:
        return round(value / 10) * 10
    elif abs(value) >= 10:
        return round(value)
    else:
        return round(value, 1)


def get_auto_color_range(values, lower_q=0.02, upper_q=0.98):
    """
    Catchall color scaling:
    - Uses the 2nd and 98th percentile instead of raw min/max
    - Prevents outliers from ruining the map
    - Keeps hover values unchanged
    """
    values = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()

    if values.empty:
        return None, None, None, None

    raw_min = values.min()
    raw_max = values.max()

    color_min = values.quantile(lower_q)
    color_max = values.quantile(upper_q)

    color_min = nice_round(color_min)
    color_max = nice_round(color_max)

    # Avoid broken range when all values are similar
    if color_min == color_max:
        color_min = raw_min
        color_max = raw_max

    # For mostly positive variables, keep the bottom at 0 if close enough
    if raw_min >= 0 and color_min > 0:
        color_min = 0

    # Tick labels
    tickvals = np.linspace(color_min, color_max, 5)

    ticktext = []
    for i, v in enumerate(tickvals):
        v_clean = nice_round(v)

        if i == len(tickvals) - 1 and raw_max > color_max:
            ticktext.append(f"≥{v_clean:g}")
        elif i == 0 and raw_min < color_min:
            ticktext.append(f"≤{v_clean:g}")
        else:
            ticktext.append(f"{v_clean:g}")

    return color_min, color_max, tickvals, ticktext

In [ ]:
def make_county_map_from_recipe(county_long, counties_geojson, recipe):
    variable = recipe["variable"]
    transform = recipe.get("transform", "none")
    animated = recipe.get("animated", False)

    legend = get_legend_settings(variable, transform)
    clean_label = get_variable_label(variable)

    # -----------------------------
    # 1. Filter by variable/year
    # -----------------------------
    if animated:
        year_start = recipe.get("year_start") or county_long["year"].min()
        year_end = recipe.get("year_end") or recipe.get("year") or county_long["year"].max()

        map_df = county_long[
            (county_long["variable"] == variable) &
            (county_long["year"] >= int(year_start)) &
            (county_long["year"] <= int(year_end))
        ].copy()
    else:
        year_start = None
        year_end = None

        map_df = county_long[
            (county_long["variable"] == variable) &
            (county_long["year"] == int(recipe["year"]))
        ].copy()

    # -----------------------------
    # 2. Filter by state
    # -----------------------------
    state_filter = recipe.get("state_filter")

    if state_filter:
        if isinstance(state_filter, list):
            state_list_lower = [s.lower() for s in state_filter]
            map_df = map_df[map_df["state_name"].str.lower().isin(state_list_lower)]
        else:
            map_df = map_df[map_df["state_name"].str.lower() == state_filter.lower()]

    # -----------------------------
    # 3. Filter by county
    # -----------------------------
    county_filter = recipe.get("county_filter")

    if county_filter:
        if isinstance(county_filter, list):
            county_list_lower = [c.lower() for c in county_filter]
            map_df = map_df[
                map_df["county"].str.lower().isin(county_list_lower) |
                map_df["county_name"].str.lower().isin(county_list_lower)
            ]
        else:
            county_lower = county_filter.lower()
            map_df = map_df[
                (map_df["county"].str.lower() == county_lower) |
                (map_df["county_name"].str.lower() == county_lower)
            ]

    # -----------------------------
    # 4. Clean missing/sentinel values
    # -----------------------------
    map_df["value"] = pd.to_numeric(map_df["value"], errors="coerce")
    map_df.loc[map_df["value"] < -1000000, "value"] = np.nan
    map_df = map_df.replace([np.inf, -np.inf], np.nan)
    map_df = map_df.dropna(subset=["value"])

    if map_df.empty:
        raise ValueError(
            "No data matched this recipe after cleaning missing values. "
            "Check variable, year, state_filter, or county_filter.\n"
            f"Recipe: {recipe}"
        )



    # -----------------------------
    # 6. Clean hover labels
    # -----------------------------
    map_df["hover_variable"] = clean_label
    map_df["hover_value"] = map_df["value"].apply(
        lambda x: format_value_for_hover(x, variable)
    )

    # -----------------------------
    # 7. Auto color range using quantiles
    # -----------------------------
    color_min, color_max, tickvals, ticktext = get_auto_color_range(
        map_df["value"]
    )

    range_color = None
    if color_min is not None and color_max is not None:
        range_color = [color_min, color_max]

    # -----------------------------
    # 8. Build title
    # -----------------------------
    if recipe.get("title"):
        title = recipe["title"]
    else:
        if animated:
            title = f"{clean_label} by County, {year_start}-{year_end}"
        else:
            title = f"{clean_label} by County, {recipe['year']}"

    # -----------------------------
    # 9. Build choropleth
    # -----------------------------
    common_args = dict(
        data_frame=map_df,
        geojson=counties_geojson,
        locations="GEOID",
        color="value",
        range_color=range_color,
        custom_data=["year", "state_name", "hover_variable", "hover_value"],
        hover_name="county_name",
        hover_data={},
        color_continuous_scale=recipe.get("color_scale", "Viridis"),
        scope="usa",
        title=title
    )

    if animated:
        fig = px.choropleth(
            **common_args,
            animation_frame="year"
        )
    else:
        fig = px.choropleth(**common_args)

    fig.update_traces(
        hovertemplate=(
            "<b>%{hovertext}</b><br>"
            "Year: %{customdata[0]}<br>"
            "State: %{customdata[1]}<br>"
            "%{customdata[2]}: %{customdata[3]}"
            "<extra></extra>"
        )
    )

    # -----------------------------
    # 10. Zoom to selected geography
    # -----------------------------
    if state_filter or county_filter:
        fig.update_geos(
            fitbounds="locations",
            visible=False,
            projection_scale=1.15
        )
    else:
        fig.update_geos(
            visible=False
        )

    # -----------------------------
    # 11. Improve animation controls and map size
    # -----------------------------
    if animated:
        fig.update_layout(
            height=720,
            geo=dict(
                domain=dict(
                    x=[0.02, 0.84],
                    y=[0.20, 1.0]
                )
            )
        )

        if fig.layout.sliders:
            fig.layout.sliders[0].update(
                x=0.18,
                y=0.06,
                len=0.55,
                xanchor="left",
                yanchor="bottom",
                currentvalue=dict(
                    prefix="Year: ",
                    font=dict(size=16)
                ),
                pad=dict(t=30, b=10)
            )

        if fig.layout.updatemenus:
            fig.layout.updatemenus[0].update(
                x=0.04,
                y=0.06,
                xanchor="left",
                yanchor="bottom",
                direction="left",
                pad=dict(r=10, t=10),
                showactive=False
            )

    else:
        fig.update_layout(
            height=650,
            geo=dict(
                domain=dict(
                    x=[0.02, 0.86],
                    y=[0.02, 1.0]
                )
            )
        )

    # -----------------------------
    # 12. Improve legend/colorbar
    # -----------------------------
    colorbar_settings = dict(
        title=legend["title"],
        thickness=18,
        len=0.70,
        x=0.90,
        y=0.55
    )

    if tickvals is not None and ticktext is not None:
        colorbar_settings["tickvals"] = tickvals
        colorbar_settings["ticktext"] = ticktext
    else:
        colorbar_settings["tickformat"] = legend["tickformat"]

    # -----------------------------
    # 13. Final layout polish
    # -----------------------------
    fig.update_layout(
        margin={"r": 20, "t": 70, "l": 20, "b": 20},
        coloraxis_colorbar=colorbar_settings,
        title=dict(
            x=0.03,
            xanchor="left",
            font=dict(size=22)
        )
    )

    return fig

In [ ]:
def make_stats_from_recipe(county_long, recipe):
    df = filter_county_data(county_long, recipe)

    summary = df["value"].describe()

    top = df.sort_values("value", ascending=False).head(10)[
        ["county_name", "value"]
    ]

    bottom = df.sort_values("value", ascending=True).head(10)[
        ["county_name", "value"]
    ]

    return summary, top, bottom

In [ ]:
def make_time_series_from_recipe(county_long, recipe, county_name=None):
    variable = recipe["variable"]

    df = county_long[county_long["variable"] == variable].copy()
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df.loc[df["value"] < -1000000, "value"] = np.nan
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["value"])

    county_filter = county_name or recipe.get("county_filter")
    state_filter = recipe.get("state_filter")

    # Normalize county_filter
    if isinstance(county_filter, list):
        county_filter = county_filter[0] if len(county_filter) > 0 else None

    # Normalize state_filter
    if isinstance(state_filter, list):
        state_list = state_filter
    elif state_filter:
        state_list = [state_filter]
    else:
        state_list = []

    if county_filter:
        county_filter_lower = county_filter.lower()

        df = df[
            (df["county"].str.lower() == county_filter_lower) |
            (df["county_name"].str.lower() == county_filter_lower)
        ]

        if state_list:
            state_list_lower = [s.lower() for s in state_list]
            df = df[df["state_name"].str.lower().isin(state_list_lower)]

        title = f"{variable} over time in {county_filter}"

    elif state_list:
        state_list_lower = [s.lower() for s in state_list]
        df = df[df["state_name"].str.lower().isin(state_list_lower)]

        df = (
            df.groupby(["year", "state_name"], as_index=False)["value"]
            .mean()
        )

        title = f"Average county {variable} over time"

    else:
        df = (
            df.groupby("year", as_index=False)["value"]
            .mean()
        )

        title = f"Average county {variable} over time in the United States"

    fig = px.line(
        df,
        x="year",
        y="value",
        color="state_name" if "state_name" in df.columns else None,
        markers=True,
        title=title
    )

    return fig

In [ ]:
def extract_json(text):
    text = text.strip()
    text = text.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if match:
        text = match.group(0)

    return json.loads(text)

In [ ]:
def build_dashboard_recipe_prompt(user_request, county_long, current_recipe):
    available_variables = sorted(
        county_long["variable"].dropna().unique()
    )

    available_years = sorted(
        int(year)
        for year in county_long["year"].dropna().unique()
    )

    available_states = sorted(
        county_long["state_name"].dropna().unique()
    )

    prompt = f"""
You are the AI controller for a county-level ACS Census and FEMA risk dashboard.

The app already has a cleaned county-level ACS Census and FEMA National Risk Index dataset.
The user does not upload data.
Your job is to convert the user's request into a dashboard recipe.

Current dashboard recipe:
{json.dumps(current_recipe, indent=2)}

Available variables:
{available_variables}

Available years:
{available_years}

Available states:
{available_states}

User request:
{user_request}

Return only valid JSON.
Do not return markdown.
Do not explain your answer.
Do not include comments.

Schema:
{{
  "intent": "make_map" | "make_stats" | "make_timeseries" | "answer_question",
  "variable": one of the available variables,
  "year": one of the available years or null,
  "year_start": one of the available years or null,
  "year_end": one of the available years or null,
  "animated": true or false,
  "state_filter": one state name as a string, multiple state names as a list of strings, or null,
  "county_filter": one county name as a string, multiple county names as a list of strings, or null,
  "color_scale": one of ["Viridis", "Plasma", "Inferno", "Magma", "Cividis", "Blues", "Greens", "Reds", "Purples", "Oranges", "YlOrRd", "YlGnBu", "RdBu"],
  "title": string
}}

Intent rules:
- If the user asks to map, show, display, color, visualize, see counties, or make a choropleth, use "make_map".
- If the user asks for top, bottom, highest, lowest, average, mean, median, ranking, rank, summary, descriptive statistics, distribution, maximum, minimum, or outliers, use "make_stats".
- If the user asks for trend, change over time, history, increase, decrease, timeline, time series, line chart, or over the years, use "make_timeseries".
- If the user asks why, explain, interpret, what does this mean, what pattern, compare, whether a value is normal, or asks about the current map, use "answer_question".
- If a request could be either a map or statistics, prefer "make_map" when the user asks to see geographic patterns and "make_stats" when the user asks to rank or summarize counties.

General variable rules:
- Choose the closest exact variable from the available variables.
- Never invent a variable that is not in the available variables list.
- Keep the current variable if the user does not clearly request a different one.
- Prefer percentage variables when the user asks for a rate, share, proportion, or percentage.
- Prefer population or unit-count variables when the user asks how many people, households, or housing units.
- Prefer FEMA risk scores for relative risk comparisons.
- Prefer FEMA Expected Annual Loss variables for financial-loss questions.
- Prefer FEMA population-exposure variables for questions about how many people are exposed.
- Prefer FEMA annual-frequency variables for questions about how often a hazard occurs.

Demographic variable rules:
- If the user says population or total population, use "total_population".
- If the user says age or median age, use "median_age".
- If the user says children, minors, youth, under 18, or child population:
  - Use "under_18_pct" for a percentage or rate request.
  - Use "under_18_population" for a count request.
- If the user says seniors, elderly, older adults, age 65 or older, or 65 plus:
  - Use "age_65_plus_pct" for a percentage or rate request.
  - Use "age_65_plus_population" for a count request.

Employment and labor rules:
- If the user says unemployment, unemployed, or jobless, use "unemployment_rate".
- If the user says employment rate or percentage of the labor force employed, use "employment_rate".
- If the user says employment-to-population ratio, working-age population employed, or share of adults employed, use "employment_population_ratio".
- If the user says labor-force participation, labor participation, or participation rate, use "labor_force_participation_rate".
- If the user asks for the number employed, use "employed".
- If the user asks for the number unemployed, use "unemployed".
- If the user asks for the labor force count, use "labor_force".
- If the user asks for the civilian labor force, use "civilian_labor_force".

Income and poverty rules:
- If the user says income or household income, use "median_household_income".
- If the user says per-person income, income per resident, or per capita income, use "per_capita_income".
- If the user says poverty, poor, or poverty rate, use "poverty_rate".
- If the user says income inequality, inequality, wealth inequality, or Gini, use "gini_index".

Education rules:
- If the user says bachelor's, college degree, degree attainment, university degree, or higher education, use "bachelors_or_higher_pct".
- If the user says high school graduates or highest education is high school, use "high_school_grad_pct".
- If the user says less than high school, no diploma, without a diploma, or low educational attainment, use "less_than_high_school_pct".
- If the user only says high school and the meaning is ambiguous, prefer "high_school_grad_pct".

Housing rules:
- If the user says housing units, homes, or total housing stock, use "housing_units".
- If the user says occupied housing units or occupied homes, use "occupied_housing_units".
- If the user says vacant housing units or number of vacant homes, use "vacant_housing_units".
- If the user says vacancy, vacant homes, housing vacancy, or vacancy rate, use "vacancy_rate".
- If the user says owner-occupied homes or number of homeowners, use "owner_occupied_units".
- If the user says renter-occupied homes or number of renters, use "renter_occupied_units".
- If the user says homeownership rate, owner occupancy, or percentage of homeowners, use "owner_occupied_pct".
- If the user says renter rate, renter occupancy, or percentage of renters, use "renter_occupied_pct".
- If the user says home value, property value, housing value, or house price, use "median_home_value".
- If the user says rent, rental cost, or gross rent, use "median_gross_rent".
- If the user says housing age, age of homes, construction year, or year built, use "median_year_built".

Transportation rules:
- If the user says no vehicle, no car, households without cars, car access, vehicle access, transportation vulnerability, or evacuation access:
  - Use "no_vehicle_pct" for a percentage or rate request.
  - Use "households_no_vehicle" for a count request.
- If the user asks for the total household universe used for vehicle availability, use "households_vehicle_universe".

Overall FEMA risk rules:
- If the user says FEMA risk, overall disaster risk, natural-hazard risk, or overall hazard risk:
  - Prefer "fema_risk_score" for general comparisons.
  - Use "fema_risk_value" when the user explicitly asks for the underlying risk value.
  - Use "fema_risk_percentile" when the user asks for percentile or national rank.
  - Use "fema_risk_rating" when the user asks for the FEMA category or rating.
- If the user says expected annual loss, annual disaster loss, average yearly hazard loss, or EAL:
  - Prefer "fema_expected_annual_loss" for estimated dollars.
  - Use "fema_eal_score" for relative comparisons.
  - Use "fema_eal_percentile" for percentile or national rank.
  - Use "fema_eal_rating" for the FEMA category or rating.

FEMA social vulnerability rules:
- If the user says social vulnerability, vulnerable population, disaster vulnerability, or community vulnerability:
  - Prefer "fema_social_vulnerability_score".
  - Use "fema_social_vulnerability_percentile" for percentile or national rank.
  - Use "fema_social_vulnerability_rating" for the FEMA category or rating.

FEMA resilience rules:
- If the user says resilience, community resilience, disaster preparedness, recovery capacity, or ability to recover:
  - Prefer "fema_resilience_score".
  - Use "fema_resilience_value" when the user explicitly asks for the underlying resilience value.
  - Use "fema_resilience_percentile" for percentile or national rank.
  - Use "fema_resilience_rating" for the FEMA category or rating.

Combined flood rules:
- If the user says flood risk, flooding, total flood risk, combined flood risk, or flooding in general without specifying inland or coastal:
  - Prefer "fema_flood_max_risk_score" for a general relative-risk comparison.
  - Use "fema_flood_expected_annual_loss" for combined financial loss.
  - Use "fema_flood_population_exposure" for combined exposed population.
- If the user asks which counties face the greatest overall flood risk, use "fema_flood_max_risk_score".
- If the user asks about flood losses, flood damage, financial flood risk, or annual flood cost, use "fema_flood_expected_annual_loss".
- If the user asks how many people are exposed to flooding, use "fema_flood_population_exposure".

Coastal flood rules:
- If the user says coastal flood, coastal flooding, storm-surge flooding, or coastal inundation:
  - Prefer "fema_coastal_flood_risk_score" for relative risk.
  - Use "fema_coastal_flood_risk_rating" for the FEMA category.
  - Use "fema_coastal_flood_risk_value" for the underlying risk value.
  - Use "fema_coastal_flood_expected_annual_loss" for total annual loss.
  - Use "fema_coastal_flood_eal_score" for relative annual-loss comparisons.
  - Use "fema_coastal_flood_eal_rating" for the annual-loss category.
  - Use "fema_coastal_flood_population_exposure" for exposed population.
  - Use "fema_coastal_flood_building_exposure" for exposed buildings or building value.
  - Use "fema_coastal_flood_total_exposure" for total exposure.
  - Use "fema_coastal_flood_annual_frequency" for annual frequency.
  - Use "fema_coastal_flood_events" for event counts.
  - Use "fema_coastal_flood_exposed_area" for geographic area exposed.
  - Use "fema_coastal_flood_building_eal" for expected annual building loss.
  - Use "fema_coastal_flood_population_eal" for expected annual population loss.
  - Use "fema_coastal_flood_building_loss_rate" for building loss rate.
  - Use "fema_coastal_flood_population_loss_rate" for population loss rate.
  - Use "fema_coastal_flood_loss_rate_percentile" for the national loss-rate percentile.

Inland flood rules:
- If the user says inland flood, inland flooding, river flooding, riverine flooding, flash flooding, or non-coastal flooding:
  - Prefer "fema_inland_flood_risk_score" for relative risk.
  - Use "fema_inland_flood_risk_rating" for the FEMA category.
  - Use "fema_inland_flood_risk_value" for the underlying risk value.
  - Use "fema_inland_flood_expected_annual_loss" for total annual loss.
  - Use "fema_inland_flood_eal_score" for relative annual-loss comparisons.
  - Use "fema_inland_flood_eal_rating" for the annual-loss category.
  - Use "fema_inland_flood_population_exposure" for exposed population.
  - Use "fema_inland_flood_building_exposure" for exposed buildings or building value.
  - Use "fema_inland_flood_agriculture_exposure" for agricultural exposure.
  - Use "fema_inland_flood_total_exposure" for total exposure.
  - Use "fema_inland_flood_annual_frequency" for annual frequency.
  - Use "fema_inland_flood_events" for event counts.
  - Use "fema_inland_flood_exposed_area" for geographic area exposed.
  - Use "fema_inland_flood_building_eal" for expected annual building loss.
  - Use "fema_inland_flood_population_eal" for expected annual population loss.
  - Use "fema_inland_flood_agriculture_eal" for expected annual agricultural loss.
  - Use "fema_inland_flood_building_loss_rate" for building loss rate.
  - Use "fema_inland_flood_population_loss_rate" for population loss rate.
  - Use "fema_inland_flood_agriculture_loss_rate" for agricultural loss rate.
  - Use "fema_inland_flood_loss_rate_percentile" for the national loss-rate percentile.

Year rules:
- If the user gives one year, set "year" to that year, "year_start" to null, "year_end" to null, and "animated" to false.
- If the user gives a year range such as 2015-2020, 2015 to 2020, or from 2015 through 2020, set "year_start" to the first year, "year_end" to the last year, "year" to the last year, and "animated" to true for maps.
- If the user asks to animate, show over time, map over time, or show change from one year to another, set "animated" to true.
- If the intent is "make_timeseries", set "year_start" and "year_end" to the requested range when provided.
- If the intent is "make_timeseries" and no range is provided, use the earliest and latest available years.
- If no year is specified for a map or statistics request, use the newest available year.
- Only use years from the available years list.
- FEMA variables are generally static county-level values repeated across ACS years.
- For a FEMA variable, if the user does not request a year or trend, use the newest available year and set "animated" to false.
- Do not create a FEMA time series unless the user explicitly asks for one.
- If a user asks for change over time in a FEMA variable, retain the requested intent but do not imply that repeated FEMA values represent historical yearly FEMA measurements.

Geography rules:
- If the user names one state, set "state_filter" to that exact available state name as a string.
- If the user names multiple states, set "state_filter" to a list of exact available state names.
- If the user says all states, United States, nationwide, national, countrywide, America, USA, U.S., or US, set "state_filter" to null.
- If the user names one county, set "county_filter" to that county name as a string, for example "Cuyahoga County".
- If the user names multiple counties, set "county_filter" to a list of county-name strings.
- If a county name could exist in multiple states and the user provides a state, set both county_filter and state_filter.
- If no geography is specified, keep the current state_filter and county_filter unless the request clearly asks for nationwide data.
- If the user asks to clear, remove, reset, or show all geographic filters, set both state_filter and county_filter to null.

Color-scale rules:
- If the user requests a specific color, use the closest valid Plotly color scale.
- If the user requests red, use "Reds".
- If the user requests green, use "Greens".
- If the user requests blue, use "Blues".
- If the user requests purple, use "Purples".
- If the user requests orange, use "Oranges".
- If the user requests yellow-orange-red, use "YlOrRd".
- If the user requests yellow-green-blue, use "YlGnBu".
- Do not use "RdBu" unless the variable has a meaningful central or neutral value.
- Explicit user color requests override default variable color rules.

ACS color rules:
- Use "Plasma" for total_population.
- Use "Purples" for median_age, under_18_population, under_18_pct, age_65_plus_population, and age_65_plus_pct.
- Use "Reds" for poverty_rate, unemployment_rate, vacancy_rate, households_no_vehicle, no_vehicle_pct, and less_than_high_school_pct.
- Use "Greens" for employment_rate, employment_population_ratio, labor_force_participation_rate, median_household_income, per_capita_income, median_home_value, and median_gross_rent.
- Use "Blues" for high_school_grad_pct and bachelors_or_higher_pct.
- Use "Oranges" for gini_index.
- Use "Cividis" for housing_units, occupied_housing_units, vacant_housing_units, owner_occupied_units, renter_occupied_units, and households_vehicle_universe.
- Use "Blues" for owner_occupied_pct.
- Use "Oranges" for renter_occupied_pct.
- Use "Viridis" for median_year_built and any ACS variable without a more specific rule.

FEMA overall color rules:
- Use "Reds" for fema_risk_value, fema_risk_score, fema_risk_percentile, and fema_risk_rating.
- Use "Oranges" for fema_eal_score, fema_eal_percentile, fema_eal_rating, and fema_expected_annual_loss.
- Use "Purples" for fema_social_vulnerability_score, fema_social_vulnerability_percentile, and fema_social_vulnerability_rating.
- Use "Greens" for fema_resilience_value, fema_resilience_score, fema_resilience_percentile, and fema_resilience_rating.

FEMA flood color rules:
- Use "YlOrRd" for combined, coastal, and inland flood risk scores, risk values, risk ratings, Expected Annual Loss values, EAL scores, EAL ratings, and loss rates.
- Use "Blues" for coastal and inland flood annual frequency, event counts, exposed area, population exposure, building exposure, agricultural exposure, and total exposure.
- Use "Oranges" for combined flood Expected Annual Loss.
- Use "Purples" for combined flood population exposure.
- Use "Reds" for fema_flood_max_risk_score.
- For FEMA risk, loss, exposure, frequency, and social-vulnerability variables, higher values should appear darker because higher values represent greater hazard, exposure, loss, or concern.
- For FEMA resilience variables, higher values should appear darker green because higher values represent greater resilience.

Title rules:
- Make the title readable, specific, and concise.
- Convert variable names into readable title case rather than displaying raw underscore-separated names.
- Include the variable, geography, and year or year range.
- For animated maps, include the year range.
- For multiple states, mention the state names in the title.
- For nationwide maps, use "United States" or "U.S." in the title.
- Use "Coastal Flood" rather than "CFLD" in titles.
- Use "Inland Flood" rather than "IFLD" in titles.
- Use "Expected Annual Loss" rather than only "EAL" unless the user used the abbreviation.
- Use "FEMA" in the title for FEMA variables.
- For combined flood variables, use wording such as "Combined Coastal and Inland Flood Risk".
- Do not claim that a FEMA map represents historical change merely because a year field is present.

Output rules:
- Return every schema key.
- Do not invent variables, years, states, counties, or color scales.
- Return null for filters that do not apply.
- Return boolean values as true or false, not strings.
- Return years as numbers, not strings.
- Use only exact variable names from the available variables list.
- Use only exact state names from the available states list.
- Ensure year_start is not later than year_end.
- When animated is false and no range is required, set year_start and year_end to null.
- The response must be parseable by json.loads().
"""
    return prompt

In [ ]:
from google import genai
import os

gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
from pathlib import Path
import re


def safe_filename(text):
    text = str(text or "dashboard").strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "dashboard"


def dataframe_to_html_table(df, variable=None, max_rows=10):
    """
    Converts DataFrame or Series to a clean standalone HTML table.
    Used for saving the full dashboard HTML.
    """
    if df is None:
        return ""

    if isinstance(df, pd.Series):
        out = df.reset_index()
        out.columns = ["Statistic", "Value"]
    else:
        out = df.copy()

    for col in out.columns:
        col_lower = str(col).lower()

        if col_lower == "statistic":
            out[col] = out[col].astype(str).str.replace("_", " ").str.title()

        elif col_lower in ["value", "mean", "median", "min", "max", "std", "q1", "q3"]:
            out[col] = out[col].apply(lambda x: format_value(x, variable))

        elif pd.api.types.is_numeric_dtype(out[col]):
            if col_lower in ["year"]:
                out[col] = out[col].astype("Int64").astype(str)
            elif col_lower not in ["state", "county", "geoid"]:
                out[col] = out[col].apply(lambda x: format_value(x, variable))

    return out.head(max_rows).to_html(index=False, classes="dash-table", border=0)


def summary_cards_html(plot_df, variable):
    s = pd.to_numeric(plot_df["value"], errors="coerce").dropna()

    if s.empty:
        return """
        <div class="warning-card">
            No numeric data available for this selection.
        </div>
        """

    cards = [
        ("Counties", f"{len(s):,}"),
        ("Average", format_value(s.mean(), variable)),
        ("Median", format_value(s.median(), variable)),
        ("Minimum", format_value(s.min(), variable)),
        ("Maximum", format_value(s.max(), variable)),
        ("Std. Dev.", format_value(s.std(), variable)),
    ]

    html = '<div class="summary-grid">'
    for label, value in cards:
        html += f"""
        <div class="summary-card">
            <div class="summary-label">{label}</div>
            <div class="summary-value">{value}</div>
        </div>
        """
    html += "</div>"
    return html


def save_full_dashboard_html(
    fig_map,
    fig_ts,
    summary,
    top,
    bottom,
    plot_df,
    recipe,
    output_dir=r"C:\SDSU_SCIL_CensusData\output",
):
    """
    Saves the entire dashboard as one standalone HTML file:
    header, map, time series, summary cards, and top/bottom tables.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    variable = recipe.get("variable", "variable")
    year = recipe.get("year", "year")
    state_filter = recipe.get("state_filter") or "United States"

    variable_label = nice_variable_label(variable)
    title = recipe.get(
        "title",
        f"{variable_label} in {state_filter} ({year})"
    )

    filename = safe_filename(f"dashboard_{variable}_{state_filter}_{year}") + ".html"
    output_path = output_dir / filename

    map_html = fig_map.to_html(
        include_plotlyjs="cdn",
        full_html=False,
        auto_play=False,
    )

    ts_html = fig_ts.to_html(
        include_plotlyjs=False,
        full_html=False,
        auto_play=False,
    )

    summary_html = dataframe_to_html_table(summary, variable=variable, max_rows=20)
    top_html = dataframe_to_html_table(top, variable=variable, max_rows=10)
    bottom_html = dataframe_to_html_table(bottom, variable=variable, max_rows=10)
    cards_html = summary_cards_html(plot_df, variable)

    html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>{title}</title>
    <style>
        body {{
            font-family: Arial, Helvetica, sans-serif;
            margin: 0;
            padding: 24px;
            background: #f9fafb;
            color: #111827;
        }}

        .dashboard-container {{
            max-width: 1400px;
            margin: 0 auto;
        }}

        .header-card {{
            border: 1px solid #dbeafe;
            border-radius: 16px;
            padding: 22px 24px;
            margin-bottom: 22px;
            background: linear-gradient(135deg, #eff6ff 0%, #ffffff 70%);
            box-shadow: 0 2px 8px rgba(30,64,175,0.08);
        }}

        .eyebrow {{
            font-size: 13px;
            text-transform: uppercase;
            letter-spacing: .08em;
            color: #2563eb;
            font-weight: 700;
        }}

        h1 {{
            margin: 6px 0 8px 0;
            font-size: 30px;
            color: #111827;
        }}

        h2 {{
            margin-top: 32px;
            margin-bottom: 6px;
            color: #111827;
        }}

        h3 {{
            margin-top: 22px;
            margin-bottom: 10px;
            color: #111827;
        }}

        .subtle {{
            color: #6b7280;
            margin-bottom: 10px;
        }}

        .section-card {{
            background: white;
            border: 1px solid #e5e7eb;
            border-radius: 14px;
            padding: 18px;
            margin-bottom: 22px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.04);
        }}

        .summary-grid {{
            display: flex;
            flex-wrap: wrap;
            gap: 12px;
            margin: 10px 0 18px 0;
        }}

        .summary-card {{
            flex: 1 1 150px;
            border: 1px solid #e5e7eb;
            border-radius: 12px;
            padding: 14px 16px;
            background: white;
            box-shadow: 0 1px 2px rgba(0,0,0,0.04);
        }}

        .summary-label {{
            font-size: 12px;
            text-transform: uppercase;
            letter-spacing: .05em;
            color: #6b7280;
        }}

        .summary-value {{
            font-size: 24px;
            font-weight: 700;
            color: #111827;
            margin-top: 4px;
        }}

        .table-grid {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 24px;
            align-items: start;
        }}

        .dash-table {{
            border-collapse: collapse;
            width: 100%;
            font-size: 14px;
            margin-bottom: 12px;
            background: white;
        }}

        .dash-table th {{
            background-color: #f3f4f6;
            color: #111827;
            font-weight: 700;
            border-bottom: 1px solid #d1d5db;
            padding: 8px;
            text-align: left;
        }}

        .dash-table td {{
            padding: 8px;
            border-bottom: 1px solid #e5e7eb;
            text-align: left;
        }}

        .warning-card {{
            padding: 14px;
            border: 1px solid #fecaca;
            background: #fef2f2;
            border-radius: 10px;
        }}

        .saved-note {{
            margin-top: 18px;
            font-size: 12px;
            color: #6b7280;
        }}

        @media (max-width: 900px) {{
            .table-grid {{
                grid-template-columns: 1fr;
            }}
        }}
    </style>
</head>
<body>
    <div class="dashboard-container">

        <div class="header-card">
            <div class="eyebrow">ACS County Dashboard</div>
            <h1>{title}</h1>
            <div>
                <b>Variable:</b> {variable_label}
                &nbsp; | &nbsp;
                <b>Year:</b> {year}
                &nbsp; | &nbsp;
                <b>Area:</b> {state_filter}
                &nbsp; | &nbsp;
                <b>Rows:</b> {len(plot_df):,}
            </div>
        </div>

        <div class="section-card">
            <h2>Map</h2>
            <div class="subtle">Colors use automatic percentile scaling to improve contrast.</div>
            {map_html}
        </div>

        <div class="section-card">
            <h2>Time Series</h2>
            <div class="subtle">Annual trend for the selected variable.</div>
            {ts_html}
        </div>

        <div class="section-card">
            <h2>Statistics</h2>
            <div class="subtle">Summary values and county rankings for the selected map year.</div>

            {cards_html}

            <h3>Detailed Summary</h3>
            {summary_html}

            <div class="table-grid">
                <div>
                    <h3>Top 10 Counties</h3>
                    {top_html}
                </div>
                <div>
                    <h3>Bottom 10 Counties</h3>
                    {bottom_html}
                </div>
            </div>
        </div>

        <div class="saved-note">
            Saved from the ACS County Dashboard notebook.
        </div>

    </div>
</body>
</html>
"""

    output_path.write_text(html, encoding="utf-8")
    print("Saved full dashboard HTML to:", output_path)

    return output_path

In [ ]:
import numpy as np
import pandas as pd

def get_nice_color_range(values, lower_q=0.05, upper_q=0.95):
    """
    Returns a robust color range for maps.
    Uses percentiles so one weird county doesn't ruin the whole color scale.
    """
    s = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

    if s.empty:
        return None

    vmin = s.quantile(lower_q)
    vmax = s.quantile(upper_q)

    # Fallback if all values are the same or nearly the same
    if pd.isna(vmin) or pd.isna(vmax) or vmin == vmax:
        vmin = s.min()
        vmax = s.max()

    if vmin == vmax:
        buffer = abs(vmin) * 0.05 if vmin != 0 else 1
        vmin -= buffer
        vmax += buffer

    return float(vmin), float(vmax)

In [ ]:
def generate_dashboard_recipe_gemini(user_request, county_long, current_recipe):
    prompt = build_dashboard_recipe_prompt(
        user_request=user_request,
        county_long=county_long,
        current_recipe=current_recipe
    )

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt + "\n\nReturn only valid JSON. No markdown, no explanation."
    )

    recipe = extract_json(response.text)

    return recipe

In [ ]:
def apply_ai_recipe(current_recipe, ai_recipe):
    updated = current_recipe.copy()

    for key in [
        "intent",
        "variable",
        "year",
        "year_start",
        "year_end",
        "animated",
        "state_filter",
        "county_filter",
        "transform",
        "color_scale",
        "title"
    ]:
        if key in ai_recipe:
            updated[key] = ai_recipe[key]

    return updated

In [ ]:
from IPython.display import display, HTML
from pathlib import Path
import re
import numpy as np
import pandas as pd


# -----------------------------
# Formatting helpers
# -----------------------------

def infer_value_kind(variable):
    """
    Guess how a variable should be formatted.
    """
    variable = str(variable or "").lower()

    if any(x in variable for x in ["rate", "pct", "percent", "ratio"]):
        return "percent"
    if any(x in variable for x in ["income", "rent", "value", "wage", "salary", "earnings"]):
        return "dollar"
    if any(x in variable for x in ["population", "labor_force", "employed", "unemployed"]):
        return "count"

    return "number"


def format_value(x, variable=None):
    """
    Pretty formatting for dashboard values.
    """
    if pd.isna(x):
        return "—"

    kind = infer_value_kind(variable)

    try:
        x = float(x)
    except Exception:
        return str(x)

    if kind == "percent":
        return f"{x:,.1f}%"
    if kind == "dollar":
        return f"${x:,.0f}"
    if kind == "count":
        return f"{x:,.0f}"

    if abs(x) >= 1000:
        return f"{x:,.0f}"
    return f"{x:,.2f}"


def nice_variable_label(variable):
    """
    Turns snake_case variable names into readable labels.
    """
    return str(variable or "").replace("_", " ").title()


def safe_filename(text):
    """
    Creates a Windows-safe filename.
    """
    text = str(text or "dashboard").strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "dashboard"


def save_dashboard_html(fig, recipe, output_dir=r"C:\SDSU_SCIL_CensusData\output"):
    """
    Saves the interactive Plotly map as HTML.
    This is the right output format for animated/interactive maps.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    variable = recipe.get("variable", "variable")
    year = recipe.get("year", "year")
    state_filter = recipe.get("state_filter") or "usa"

    filename = safe_filename(f"{variable}_{state_filter}_{year}") + ".html"
    output_path = output_dir / filename

    fig.write_html(
        str(output_path),
        include_plotlyjs="cdn",
        full_html=True,
        auto_play=False,
    )

    print("Saved dashboard HTML to:", output_path)
    return output_path


def get_nice_color_range(values, lower_q=0.05, upper_q=0.95):
    """
    Robust catchall color range for choropleth maps.
    Uses the 5th and 95th percentiles so outliers do not flatten the map.
    """
    s = (
        pd.to_numeric(values, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if s.empty:
        return None

    vmin = s.quantile(lower_q)
    vmax = s.quantile(upper_q)

    if pd.isna(vmin) or pd.isna(vmax):
        return None

    if vmin == vmax:
        vmin = s.min()
        vmax = s.max()

    if vmin == vmax:
        buffer = abs(vmin) * 0.05 if vmin != 0 else 1
        vmin = vmin - buffer
        vmax = vmax + buffer

    return float(vmin), float(vmax)


def get_recipe_plot_df(county_long, recipe):
    """
    Recreates the filtered dataframe used for the dashboard.
    Supports one state, multiple states, or the entire US.
    """
    variable = recipe.get("variable")
    year = recipe.get("year")
    state_filter = recipe.get("state_filter")

    plot_df = county_long.copy()

    if variable is not None:
        plot_df = plot_df[plot_df["variable"] == variable]

    if year is not None:
        plot_df = plot_df[
            plot_df["year"].astype(int) == int(year)
        ]

    # -----------------------------
    # State filter
    # -----------------------------
    if state_filter not in [None, "", "United States", "USA", "US"]:

        # Allow either a single state or a list of states
        if isinstance(state_filter, str):
            states = [state_filter]
        else:
            states = list(state_filter)

        states = [str(s).strip() for s in states]

        if "state_name" in plot_df.columns:
            plot_df = plot_df[
                plot_df["state_name"].astype(str).isin(states)
            ]
        elif "state" in plot_df.columns:
            plot_df = plot_df[
                plot_df["state"].astype(str).isin(states)
            ]

    plot_df = plot_df.copy()
    plot_df["value"] = pd.to_numeric(plot_df["value"], errors="coerce")

    return plot_df.dropna(subset=["value"])


def apply_nice_map_scaling(fig_map, county_long, recipe):
    """
    Applies percentile-based color scaling to the map
    and restores readable colorbar numbers.
    """
    variable = recipe.get("variable")
    variable_label = nice_variable_label(variable)

    plot_df = get_recipe_plot_df(county_long, recipe)
    color_range = get_nice_color_range(plot_df["value"])

    if color_range is not None:
        vmin, vmax = color_range

        tickvals = [
            vmin,
            vmin + (vmax - vmin) * 0.25,
            vmin + (vmax - vmin) * 0.50,
            vmin + (vmax - vmin) * 0.75,
            vmax,
        ]

        ticktext = [format_value(x, variable) for x in tickvals]

        fig_map.update_coloraxes(
            cmin=vmin,
            cmax=vmax,
            colorbar=dict(
                title=("%" if infer_value_kind(variable) == "percent" else variable_label),
                tickmode="array",
                tickvals=tickvals,
                ticktext=ticktext,
                thickness=18,
                len=0.75,
            ),
        )

    return fig_map


def style_table(df, variable=None, max_rows=10):
    """
    Nicely styled dataframe/series display.
    Handles both DataFrames and Series.
    """
    if df is None:
        return None

    if isinstance(df, pd.Series):
        out = df.reset_index()
        out.columns = ["Statistic", "Value"]
    else:
        out = df.copy()

    for col in out.columns:
        col_lower = str(col).lower()

        if col_lower == "statistic":
            out[col] = out[col].astype(str).str.replace("_", " ").str.title()

        elif col_lower in ["value", "mean", "median", "min", "max", "std", "q1", "q3"]:
            out[col] = out[col].apply(lambda x: format_value(x, variable))

        elif pd.api.types.is_numeric_dtype(out[col]):
            if col_lower in ["year"]:
                out[col] = out[col].astype("Int64").astype(str)
            elif col_lower not in ["state", "county", "geoid"]:
                out[col] = out[col].apply(lambda x: format_value(x, variable))

    return (
        out.head(max_rows)
        .style
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#f3f4f6"),
                    ("color", "#111827"),
                    ("font-weight", "700"),
                    ("border-bottom", "1px solid #d1d5db"),
                    ("padding", "8px"),
                    ("text-align", "left"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("padding", "8px"),
                    ("border-bottom", "1px solid #e5e7eb"),
                    ("text-align", "left"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "auto"),
                    ("font-size", "14px"),
                    ("margin-bottom", "12px"),
                ],
            },
        ])
        .hide(axis="index")
    )


def make_summary_cards(plot_df, variable):
    """
    Creates dashboard metric cards from the filtered data.
    """
    s = pd.to_numeric(plot_df["value"], errors="coerce").dropna()

    if s.empty:
        display(HTML("""
        <div style="padding:14px;border:1px solid #fecaca;background:#fef2f2;border-radius:10px;">
            No numeric data available for this selection.
        </div>
        """))
        return

    cards = [
        ("Counties", f"{len(s):,}"),
        ("Average", format_value(s.mean(), variable)),
        ("Median", format_value(s.median(), variable)),
        ("Minimum", format_value(s.min(), variable)),
        ("Maximum", format_value(s.max(), variable)),
        ("Std. Dev.", format_value(s.std(), variable)),
    ]

    card_html = ""
    for label, value in cards:
        card_html += f"""
        <div style="
            flex: 1 1 150px;
            border: 1px solid #e5e7eb;
            border-radius: 12px;
            padding: 14px 16px;
            background: white;
            box-shadow: 0 1px 2px rgba(0,0,0,0.04);
        ">
            <div style="font-size:12px;text-transform:uppercase;letter-spacing:.05em;color:#6b7280;">
                {label}
            </div>
            <div style="font-size:24px;font-weight:700;color:#111827;margin-top:4px;">
                {value}
            </div>
        </div>
        """

    display(HTML(f"""
    <div style="
        display:flex;
        flex-wrap:wrap;
        gap:12px;
        margin:10px 0 18px 0;
    ">
        {card_html}
    </div>
    """))


# -----------------------------
# Main dashboard
# -----------------------------

def show_dashboard_from_recipe(county_long, counties_geojson, recipe):
    """
    Polished notebook dashboard:
    1. Map with robust color scaling
    2. Time series
    3. Summary statistic cards
    4. Styled top/bottom county tables
    5. Saves the full interactive dashboard as HTML
    """

    variable = recipe.get("variable")
    year = recipe.get("year")
    state_filter = recipe.get("state_filter") or "United States"

    title = recipe.get(
        "title",
        f"{nice_variable_label(variable)} in {state_filter} ({year})"
    )

    variable_label = nice_variable_label(variable)
    plot_df = get_recipe_plot_df(county_long, recipe)

    # -----------------------------
    # Header
    # -----------------------------
    display(HTML(f"""
    <div style="
        border: 1px solid #dbeafe;
        border-radius: 16px;
        padding: 22px 24px;
        margin: 14px 0 22px 0;
        background: linear-gradient(135deg, #eff6ff 0%, #ffffff 70%);
        box-shadow: 0 2px 8px rgba(30,64,175,0.08);
    ">
        <div style="font-size:13px;text-transform:uppercase;letter-spacing:.08em;color:#2563eb;font-weight:700;">
            ACS County Dashboard
        </div>
        <h2 style="margin:6px 0 8px 0;color:#111827;font-size:30px;">
            {title}
        </h2>
        <div style="font-size:15px;color:#374151;">
            <b>Variable:</b> {variable_label}
            &nbsp; | &nbsp;
            <b>Year:</b> {year}
            &nbsp; | &nbsp;
            <b>Area:</b> {state_filter}
            &nbsp; | &nbsp;
            <b>Rows:</b> {len(plot_df):,}
        </div>
    </div>
    """))

    # -----------------------------
    # Map
    # -----------------------------
    display(HTML("""
    <div style="margin-top:18px;">
        <h3 style="margin-bottom:4px;color:#111827;">Map</h3>
        <div style="color:#6b7280;margin-bottom:8px;">
            Colors use automatic percentile scaling to improve contrast.
        </div>
    </div>
    """))

    fig_map = make_county_map_from_recipe(
        county_long,
        counties_geojson,
        recipe
    )

    fig_map = apply_nice_map_scaling(
        fig_map,
        county_long,
        recipe
    )

    fig_map.update_layout(
        margin=dict(l=10, r=10, t=60, b=10),
        title=dict(
            text=title,
            font=dict(size=24),
            x=0.01,
            xanchor="left",
        ),
    )

    fig_map.show()

    # -----------------------------
    # Time series
    # -----------------------------
    display(HTML("""
    <div style="margin-top:28px;">
        <h3 style="margin-bottom:4px;color:#111827;">Time Series</h3>
        <div style="color:#6b7280;margin-bottom:8px;">
            Annual trend for the selected variable.
        </div>
    </div>
    """))

    fig_ts = make_time_series_from_recipe(
        county_long,
        recipe
    )

    fig_ts.update_layout(
        margin=dict(l=10, r=10, t=50, b=40),
        title=dict(
            text=f"{variable_label} Over Time",
            font=dict(size=22),
            x=0.01,
            xanchor="left",
        ),
        xaxis_title="Year",
        yaxis_title=variable_label,
        hovermode="x unified",
    )

    fig_ts.show()

    # -----------------------------
    # Statistics
    # -----------------------------
    display(HTML("""
    <div style="margin-top:28px;">
        <h3 style="margin-bottom:4px;color:#111827;">Statistics</h3>
        <div style="color:#6b7280;margin-bottom:8px;">
            Summary values and county rankings for the selected map year.
        </div>
    </div>
    """))

    make_summary_cards(plot_df, variable)

    summary, top, bottom = make_stats_from_recipe(
        county_long,
        recipe
    )

    display(HTML("""
    <h4 style="margin-top:18px;margin-bottom:8px;color:#111827;">Detailed Summary</h4>
    """))
    display(style_table(summary, variable=variable, max_rows=20))

    display(HTML("<h4 style='margin-top:18px;color:#111827;'>Top 10 Counties</h4>"))
    display(style_table(top, variable=variable, max_rows=10))

    display(HTML("<h4 style='margin-top:18px;color:#111827;'>Bottom 10 Counties</h4>"))
    display(style_table(bottom, variable=variable, max_rows=10))

    # -----------------------------
    # Save full dashboard HTML
    # -----------------------------
    saved_html_path = save_full_dashboard_html(
        fig_map=fig_map,
        fig_ts=fig_ts,
        summary=summary,
        top=top,
        bottom=bottom,
        plot_df=plot_df,
        recipe=recipe,
    )

    display(HTML(f"""
    <div style="
        margin: 18px 0 20px 0;
        padding: 10px 12px;
        border: 1px solid #d1fae5;
        background: #ecfdf5;
        border-radius: 10px;
        color: #065f46;
        font-size: 14px;
    ">
        Saved full interactive dashboard HTML to:<br>
        <code>{saved_html_path}</code>
    </div>
    """))

    return {
        "map": fig_map,
        "time_series": fig_ts,
        "summary": summary,
        "top": top,
        "bottom": bottom,
        "plot_df": plot_df,
        "html_path": saved_html_path,
    }

## Testing Area

### Currently Supported Data:

#### Demographics

**total_population** — Total number of people living in the county.  
**median_age** — Median age of county residents in years.  
**under_18_population** — Total number of residents under age 18.  
**under_18_pct** — Percentage of residents under age 18.  
**age_65_plus_population** — Total number of residents age 65 and older.  
**age_65_plus_pct** — Percentage of residents age 65 and older.  

#### Employment & Labor Force

**employment_population_ratio** — Percentage of the working-age population that is employed.  
**labor_force_participation_rate** — Percentage of the working-age population participating in the labor force.  
**employment_rate** — Percentage of the labor force that is employed.  
**unemployment_rate** — Percentage of the labor force that is unemployed.  

#### Income & Poverty

**median_household_income** — Median annual income earned by households in the county.  
**per_capita_income** — Average annual income per resident.  
**gini_index** — Measure of income inequality ranging from 0 (perfect equality) to 1 (perfect inequality).  
**poverty_rate** — Percentage of residents living below the federal poverty threshold.  

#### Education

**less_than_high_school_pct** — Percentage of adults without a high school diploma.  
**high_school_grad_pct** — Percentage of adults whose highest education is high school graduation.  
**bachelors_or_higher_pct** — Percentage of adults with a bachelor’s degree or higher.  

#### Housing

**housing_units** — Total number of housing units.  
**occupied_housing_units** — Total number of occupied housing units.  
**vacant_housing_units** — Total number of vacant housing units.  
**vacancy_rate** — Percentage of housing units that are vacant.  
**owner_occupied_units** — Number of owner-occupied housing units.  
**renter_occupied_units** — Number of renter-occupied housing units.  
**owner_occupied_pct** — Percentage of occupied housing units that are owner-occupied.  
**renter_occupied_pct** — Percentage of occupied housing units that are renter-occupied.  
**median_home_value** — Median value of owner-occupied homes in the county.  
**median_gross_rent** — Median monthly rent, including selected housing costs.  
**median_year_built** — Median year residential structures were built.  

#### Transportation

**households_vehicle_universe** — Total number of households used for vehicle availability estimates.  
**households_no_vehicle** — Number of households without access to a vehicle.  
**no_vehicle_pct** — Percentage of households without access to a vehicle.  

#### FEMA Overall Risk

**fema_risk_value** — Estimated overall annualized financial risk from natural hazards.  
**fema_risk_score** — Standardized overall natural-hazard risk score.  
**fema_risk_rating** — FEMA category describing the county’s overall risk level.  
**fema_risk_percentile** — County’s overall risk percentile compared with other U.S. counties.  

**fema_eal_score** — Standardized Expected Annual Loss score.  
**fema_eal_rating** — FEMA category describing the county’s Expected Annual Loss level.  
**fema_eal_percentile** — County’s Expected Annual Loss percentile compared with other counties.  
**fema_expected_annual_loss** — Estimated average annual financial loss from natural hazards.  

#### FEMA Social Vulnerability & Resilience

**fema_social_vulnerability_score** — Standardized measure of population vulnerability to disasters.  
**fema_social_vulnerability_rating** — FEMA category describing the county’s social vulnerability.  
**fema_social_vulnerability_percentile** — County’s social vulnerability percentile compared with other counties.  

**fema_resilience_score** — Standardized measure of the county’s ability to prepare for and recover from hazards.  
**fema_resilience_rating** — FEMA category describing the county’s community resilience.  
**fema_resilience_percentile** — County’s resilience percentile compared with other counties.  
**fema_resilience_value** — Underlying FEMA community-resilience index value.  

#### FEMA Coastal Flood Risk

**fema_coastal_flood_events** — Estimated annual number of coastal flood events.  
**fema_coastal_flood_annual_frequency** — Average annual frequency of coastal flooding.  
**fema_coastal_flood_population_exposure** — Population exposed to coastal flooding.  
**fema_coastal_flood_expected_annual_loss** — Estimated annual financial loss from coastal flooding.  
**fema_coastal_flood_risk_score** — Standardized FEMA coastal flood risk score.  
**fema_coastal_flood_risk_rating** — FEMA categorical rating for coastal flood risk.  

#### FEMA Inland Flood Risk

**fema_inland_flood_events** — Estimated annual number of inland flood events.  
**fema_inland_flood_annual_frequency** — Average annual frequency of inland flooding.  
**fema_inland_flood_population_exposure** — Population exposed to inland flooding.  
**fema_inland_flood_expected_annual_loss** — Estimated annual financial loss from inland flooding.  
**fema_inland_flood_risk_score** — Standardized FEMA inland flood risk score.  
**fema_inland_flood_risk_rating** — FEMA categorical rating for inland flood risk.  

#### Combined Flood Metrics

**fema_flood_expected_annual_loss** — Combined estimated annual financial loss from coastal and inland flooding.  
**fema_flood_population_exposure** — Combined population exposed to coastal and inland flooding.  
**fema_flood_max_risk_score** — Higher of the county's coastal or inland flood risk scores.

### Test Code

In [ ]:
print("done with setup")

In [ ]:
user_request = "show gini index in the usa 2010-2024"

ai_recipe = generate_dashboard_recipe_gemini(
    user_request=user_request,
    county_long=county_long,
    current_recipe=current_recipe
)

display(ai_recipe)

current_recipe = apply_ai_recipe(
    current_recipe,
    ai_recipe
)

display(current_recipe)

dashboard_output = show_dashboard_from_recipe(
    county_long,
    counties_geojson,
    current_recipe
)